# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz3DaS3+dX4JhKhYxHzIxkO44ukzrbkhPXybbOcZK9VVQszBCcocVX+Bhp5PL9rf2+v+y6GyAJ8DGSvV5XIpEg0AAa/UYDsizrVVqm+TEL4JddOOzgJxbzKOLs6ITZv1TJruLJ0cnBIfuOHcZb+l6Km7LKOT4+i0Tii9ydTH4rUuYLlotVlRdpcTxh7Fv2LvXTgkXwf5z6An//VQmfJ2zF8zXHbyJh0NfP578xjg9bccvsTORBGLFXT//28tUb9h1AYuzp2bs3zpTxImRJyjZ8x0QB9ThCKHPBqoTTFNgOwRThugrhg3BpGGfQUSZgVKwQLIMhYu+J6vvt01fMXvHVRuD4M74OE4AKj2dhUt04LBI7mGNKo8AZ8Hy1Cbdy6BnPeQTzOmYKaA4DUXNlJc99GIpYV9C8kCM5jViZ8yV/D59FnKXQnG05gvrOF9vvik3MbD8sVikWwcgcNQGJIhp9Woq1YKs0wa5WPPEBj/9ZLwSzn0O9B9QU6uSAGZHQ0OuhUksdRWwtEkDkKkyTdjVavOWiqKKS1gq6X1c4KaqIGJdok0NlMBtYmGKDuIvq1WjR9r4qSlEgpjJRhjlWicMiVhVFwcKkKHkC/4nUnViWNQkRQSUOGGgqh7G6QQWEB1VhgVZB/X29qp+w9yhc1q9hWj+9L9Kkfk6L+ikX9VOxqcowat6qJaB5JYqmZrnJBffDZN0UhHHT+DbMgFzFpH5Pqjjb4RCTbBLkaczOX54x9e1lzNdiKn+9yYqJrOCydqpBuJ5Mfjl9e8oWMFQ34+XG9cM84bGw63e+LPC37XnYsec5zgTWzDt78/y/oVUzWPcsXV3ZzuT3l6d//AofbAt6S0pryqwlX13h70gE9J6H601pOZPJZBXxomCnCbCBOKbV80XAPC9MwtLz7EJEwRSpMFi8ThOYSpSuF1keJqUja+M/rORm0CPWY0AWcl6uL0qxKmFERkWAAFXhp1mci3iJH7Ab80ux4ZkY/+LF2z0fy70NRz5nHCY4/sW7ChMfPgPNGh/fp8sCij9YfFWiyLCO2QyQLRJvlUYc3z42Db4CmkKJAKJS+CGym40PwCTpNowc4l/gEiA8EDIrEMrANAEHzjS7FCWnHoM0j7GHJy72qCQ2vB89ooKlFBdQ8HjmaqMgGB4A8ZDANBJ8n4aJrZbxzW/vvJOXbxGuHLKL/GW1y1rmu5YY9KG5VebzUthYH9ad+3aaicQ2e3WcFpK4WYmsZKf0CySUCTYDUjWHLgUSIOBjl5i4v4PyFzwqOiuoUOFlaRpBjVXgviMGOof30xtQZ6Ae7ZjfeNdpfiXyYnEIklGu/YWlyqzLdsiAjphfwdLlRQ9h4iYsSi+9WrzLKwHcJhf+4F/4JzVpw6eg3BCt3rVAhi4Uu4LITYuFDWSQrECMfqepdYuoo1ccby1HY2iQxmdCsLJR540SRJVJmpXo0xfveR41ah20gmtpTEHCblOtQbasA74S3qZa1qKvAFXm0fJ5gLq2CQg45KGLy3HqAlpnOaiPDgzbcWniZmX8FwYsp29e6GO7kWoNbLGlWtBkGxZAhSN12/E+gAEXZW4HLgloomoCFcjutlReXI7SOaoPMcBDICVt63VKlkDlp2CcCNKkjf3yy4tjWFLhGE1zAdyfsNmkKUVayX2bBmZ2kwCyZ0bJdQiSgPgUq6OyWEIXyyoIBMj99WLm4GiDPk6uNzBJhqQ+jK8ldCVZ0378kP34IzucOYMVYb2StGTLcbSrGSaDFRJcD+Bye+m0GEBxjlrcxR+aSqLZDguB+WOaqbgxx1GmJUfJUVSxLW6A+TM796eSEpy+rrMD61xyjLIZlRkMzHLMPkhgYGt/++3R7NidBx/Zz8/ws9WfWmB90GbADlh57M6Cj4UmiRVeCOqk1eZS3WVhJjQJMUXDZ8qgwTLViEIy7e7IBztR6kl4qLm2ERkn4bsXUXr9ipcgGJL1OYCOQk1ZYl+Ao7vqu9ibB5gBKxlefbtDx2qUQRqByF7QeKtCeAUPwLRIihTEsxStGunQhMwl+4q9ADNn8/sJoNwXq9QPg3BFDoj0ff5vPruBR7B+eRYq239U/NTTc0XCQZl4AYLe+rEdrzwerdOFFa8sk7AVu7/bZeI0z8H96gEc7GW0pz7b3EOg9AULa9CCThb4IVmahNDRgEi5J/xPhq0IFmfZ0itZCpJQr8HwATuNVlm9kQmmF9CC6wV/VTwKyx1ZrKZWe44MqDuoLnuHrZn9CK0LKH/QjNtBJ0VEWDMlZweGBuasRlrk9YDdxQpwtZBpG+ePPQcUoQu3zsFjA03K4yV4X9Idrr1hAN6AUj5kguAiGM1MDqe2A9UonsO8wP8zlGztoaSgoltph28uWv1g7BTuqvI5CKoyriIXCD699srg6BD4E5G2p1GSfGr1JdgWGzBZr7q1DbEibX018mcAYZ2ngLi3AoxfkevMrK8nCwuyyE2yM2osGksNxu2libfOKusSXRJUKC213K0XoG/NLRns2XBberMwYbVdMyAI00EZB06cRDQLVMX06IyiWkl+JLW447puR+4MSpWOcySNcU1DDJuNo6p47J+1Uc39Az8sD7aHB7MDQgAIAFNmf6Z8kSi4S3ANkIhBl10aM1fns9ZGrsoU2Jy4dWBdDLf2ngswhFAoJQ+nT2rgE+NMxsmOfN/PmB0AVrS35XkIqwQmcsmLzyA+5X3fa/rgn3w6BQ4SYbz9slTYImQfJXaWxTZIDqTTyMI4/VX7zDUzV2sfbd69bP/eRRtasj6d/wurZazTFF2rqkAlW6SggOcUnhxZPRkWapZEvg6uh6HtSnEzZEKfY/Oe4UwjriiOZF9ouKDOdCl62VErAvADbrMDFsxwO+uy50UX6I8n1N8evAVDREWjZ/aHb6j3b3AwzSNBlAP6Rlo033wcoLS9du/oh16Qbhih+z2LHtENi1zD9Rj2Ve827TUqsuTSKQwd7x0SDLsQXxQDg5N0JvvB17FOGO+wUw9u89WXckfuo8z3B2l1063hT7T9+jzaxj8LAUxEIVNwLcNb6SSjEQmFeqiv18GFlQYBeivW5fEdA619ODL0vVVWeappx5lDrrnD7lHbcS+fmvhi9gokWI5B+B34KehgtDszjmWIM8TIkKXcD7RlSKy2If+nXXWlFziD4bdMXwaSoE2Jbo6OuMBumdoWoMxyhqO8hkFHX655HleZPRiNeaW2JiNQhTIUc2dYpW7txQKd78ZRVd3gZ9PP/C3h6MBxHze9khBsdPDq5I7hMVulcRZG0g29EnkCS44r9fy3k6dsBwQQrtvZUPA1Wqd5WMYy/LqqTl6/ljujACPLYdR5vfvGcXORY9wWt1ajdHU/b1EFawzHwDBKdNKhykrp1Y6VdAXJywwLj285TA8o0u4Qg3T273S9+gHfoV21k5xfm2wfAzz65Cbi2rbe/vzsKYgQ+9H8cMrghwPPsylT/zkm2/l1WwTr4g87jDtVXBFFYVYI255//3jKHgKYoyN4OJzNHIr/RQsbnqds/j3+eAQ/Dh896nYEVAvMmawjhPMDNJ//gIDmCPH7FtBjKHyCYB4PgkGXU2L9Z9rOxXilL7bhSiwsXAfLAX8/AQMThBuI/k6glYKesnmYBCJHvUCSyYb+ZTmvynTFi9KW4KbML0HFLeRHFFzl/LEzHBgDzAEiF2E8xb1Rr+2hKEVWLGA10lWZC+HlAmyuCvXDYn74ZMRWRBArsGeuikUtseSrdTlVe9kwqMUaoFZlVoEkx3FaJTJGsekaty2hFrtktclBdN4K25mMGD4YwgFTGZQmsvJ9RMUnRfZZDTzGrfmUpXFYhn5tfmrx2yEFBVIXHrU16Om9vs7rcuAgp5l2q+uHAS1gGZICy2n7TOQuole91Yz5CoreUslkRAuqHTJ9Mih54NcdLWRHTQNFCbK0Y9lqjdXwFtrIgE0CXkWlTn1dcJ/ktuhzWdCS9EeDq9ed5+fuAa0wlshzDAoC4fCE1Lza8e1STq2zOhqqt+5q/bKC8iIGvpgKA/9tUdzKBu42zEuUNdBVmu/sMVElw5BUx8O44oqXIJucevuDgvdjmkS6E7MxRsWElIr07Ydt7FaF8I1Nle+wtLPXMpN7Lf/8x9Bei9xvQfb/sFYQFJF8s8157K2X31xic+uT9q6/xAYwCddEFPoecKYEQhivcW8H44/eci23Rgz5ABX2CQbTlsb6hczSWaFQB9hFmTtdylk3apc2DaFAMzXbr2+ywhU3YeCBO5KAzVoIs+qGw3pH2QbzGeCDi+qILRZM6nGyGZPM5XnOd9TwAhxK0JuXLhhYQB4/gpKc6UNv0CCbpmXbw9AM2mAuQkepsxV5SVaE5XQn1K3xVCODrzA3DFgGOBSoLV2+F5Slho5Xjlwr87gq7udooWN2Vgx2oWiFMiJgcKZNjV0xZTeFrJakya3IU5uzn9h8Zhj5uAV6U3TW6wbMiZv5lO3g924OMG4KicEpPfEbfNo1ZTtVNoAxQkKeZraNMHcElz1gc4ILvzWsXU/ZRrUwRCDOAcQ0plvYWMdh37K5O3+iWd5/DRt1gIFi1KAr/gLxD8aGpFnbLkBTX4OYAdbHpvC2kW9ObyMKmoJeQBVrz2eHD8H+gp+OyuByz56+fv73N786pGoJ4YcPpWAq/mqFbpFFYekVGyHKWlHjMxgoOYYSZFbWFFOxprh7MlXJWIYDcRJuQ1+mOW7S95hpoUJ3Mg0RfAaBCXq4O4T7ybg9FaSJn5rGfiB73svyWupMvl5ykxXGONcUw7IP6VVqEkPNGmWGRJImJeib44zymcEEOK6GCwjtTb0VJfFwlyc73cgtxBooBEYCSMQR8LykpBIVR5xqigSd3HDKKOVDgLGJ9qRAV7or6AK2bUSJhDzg6nZ6lJuSYcfDV6iS4PaD6uUw6TN0eQa49IHAsTvoxzH4fwjyUDOUEzhfrfUfhFTy/S7mlwZaEY+FjB5SHAuLkNCgHjBWcTHDFfoDGHnmzh5eduURVidhfUyiEoiFbSqQl8Ux7nSWmBa6BktCFL0kFWxNDOQSf9nW1HKc3txwdDaKlBBG8Aco+8SZkoiBApRJTanMzwkpPwe8MWEnTjtYzL4kUKkL1BtmtqydYu3uGNpW4H+YqWjYBkGB0JFi10EAtwCPOpjSiDtkBkAu8OtlY+WiekcO0IStkuFUisIKcO84vRi5YnCA+EUMD9rVan0SGBYPcyXg0BeAn/zGwzyv4r7pJPaJUB6ceAHt1M4pTgRf/GoFL/c0xV+gSypyBaJFAo4Mmcj4bDs2lvdqDQ6nV3nUet5hPbBXjY8xElLzyUVH5FdRdjRqDEzp+x5Wss261LeLMhJ8+0Ji2pVI7oHAb2Gy9pTp41dApGhje3Vze08TXE0PrAkPXIoQo96egDXb02QVgSbwNikwKypwekAX6HA2m+3px0tzdHYloXirdEO+ZdT1GtC7iDGrgDLBJWacQc9EOvpmY7mc6pP7Tv62Y8KjuIEpwieydGLCpVagMsEXHV76RLe+WcPh3aZ6ZAMh7sF9jz1TracLGAYrcmylJWSJDbnUbhDeeHJzpujQ92iDKKqXu1d/ED+H+wPZOZkvdBBBBj6k+3rYM2I1mmc/tVLmeGjNNdGhGLeVS6DdF62M6spJQm1rxFEcFKWXEnHbUFyjyMYYVh3BwvUU/pStKxg8mDwqm67neA350HEVlSHqi6tjsARQVVwpK4R6csNSxLCM0u7Qoueteql3lKVuJHiIn3lrHIF9ITcS9wTsoeCySZtTu4j7w/2XDvuzaxrVY1G7kJ0hUH6sFl3WQe7p3Oi5vOzGqxNxU9pZu0sh++ruNrTrjJmDhC4Zwr5zW0Ojw4JSFYbalfuaBdSyIyaQgFC4gYOFhgm9g+B54kibrS5hPy7A0JU4efR5ezdpuRF5Z8AmTjuD789lcDdHwe3s6BiliKrh3RyqNrCj0yQ70icKYY+FbkWclTuZ+G13XXPJVjBItdqEP+KoC3UcpqWjTwyi09rAg9PJHv7igXQlyoxY+no4mK5R0Kih1I+2YytZ6kz3BthHYdYSzyswgr2gedh1oWMG5scHNhiwBw9iSDRP1Yp+EVtWhUsbaa9iuIY1qwQ+2gufG2hXkEwPLgwamOw/FoCpkpdlbUhbGCkGIumkru7Zq27GpzMCOS8m1V9iYogsurqU+oZEw9jZMFI8pIlghPaVs+dAxlfsd5ldlAtJnfhsBzkdN3yAJzd5BGzrug6Tu5ZgpeC5ChaljPug+kXXBWqcH07hIcJiwwlFbRfgM2k8yeX4CsTz2UZbfc4VY3JGuhQacTROIO+pzBOSkxuw3+4cvDHC1kWT6WhfhrhTlcKOlH0ldqYBk2al7p+hoFHnKFHuzzWB2tAIHSg08YZqVKeOfqhkb97Fpj4ZduUCuZC8dMYrbUEnLHclWrUXx8fsh+81DG6MM2Z+FWeFjVPEdcpLD6avjggMdKSwv3E34sYPwdkpsYP54WVrBuZVotAno3YMuaU+CgkMox6JZ9Qzhbbkcz9M5u2NAXaD99O+5aA6kdJbvZA9q56BiOtxgKpazA+PHppQaiG9oJOB0O/C2lS4jRT6AgawoZNki7n7BD6mIJrEYgYVzcWHqunCes7BhuG5NZUFqGDqrnmcRe2QAMkl7uIMzEmJMpW9n4soFFvs8bFuSpNXts7RLYt4vPQ5QyF9zBQqEg9DRyGPugn/urFEuSKDvG+k4Jw3zDvpGmyNqsWsh/MLi56Nk4C0JqqmepFV5Ytel5ZMVZXPsiY96xXlxiFWwydZqZOW1GBfVWzfZfXmXW+klqiel3pTM5NvxjBmnWSN1hhXS2MDmRyC+vifKiwpV5Di0ZY+TBAX8rAqaaRjLb6lmKuhfqc+s6xXIkbR6/QNC6nE9EbEYiZgyWt6JSrRa30cjKEPyzPddZPmsx7/VzHwlved2rMbNK11mTpsWRjoRFUOgnjShdBo/RHrBHMLQWlHlUxatK0XeLS4OcbvC6XgtOVDoQp9+eFKscHCcIUXtUcshRH9lNII/lfCqDGgh03CRjK17jSKJ/i/EUvyVyOZ5K/pCH5QSNFPXUA1T62UUr+HwbTCa5mmkd28OtNGdskP8sVphZi0idWbMwodZFwNeZlqyumqlkSowXXdrS/2VUN08kQo+WHqCHdY+Co+Kz8CrVxYUARsPWYDva0vf8BdJmqEFs7V4OlSA3ArDvCEdPcc+bk8OO1R5wY8/fA0VukdmTZC6UPxkeNPtDy2bsG37dUKND7Zc2CRted9SD66WbLG3YWm9Wm9GEI7gP8Ve4sRLOKXCHz1GftangBDu/avStCNIJivgQqLF6K+8IQDY+VhmmsnqdBmPL1Qp/jRUrdPL5pT/JfKZUD6UmkQmOUNNepj/djiYYsrrz5aglsw8Gnm/jDDy0jaDr7DPrUGUsvUDZsWd45CwjGOGZfh6soGM3mJTL/CiHVBiQNp1lkry7KebnlyS2JniTt9rKhgbWJyG3zcLydJRFe9SPTla25scBLy5iNphO2RZDICYACYUlJg7L1PFjcqIgMTfzRldifLa44zxU1qOaP5gLHaqEGOq7eEVhw3m25o/v3qNJxrHpbQ4yPHxGAG9rjoozBIunwLMIz7P063gLQOBuhWh7aKPOZsl5hyUC5wraYMnovFyJJNmc9BKyaL/mEZ4B7aQLTvEcBWMiNIOpUDMCCjaOjACeKn6O2TqH6JazWsNbdvXDSXbVxi0HGuy8r6yhQQdasrzDvqJOhodswRWAmnCW4u8+P6viLgWTR3WYpkKY9rWp1AUN3DgHjtDu5AH1ynVn17SGcKo8hNtuo0uiH6SGRq8nKL2r4Y8rNIr49xUB2PovCLr+KnRKCAJ2AVJTOmmnS575adJaNuaCs2J2mPTizkH6t7vMwiY37bOvvWyMmF/j8C989/yFNaVifaDGDuO1zpehzrIVLaHtizMfA5R9YGDKL6kP7A0jWiviOxinEB1ayYdRbGWUj4x/ykXC4FX1fvRZ4W1jAdNIe/1P6vpI3OjoqxMSVucPdjSONaPF+FUcTddbS0PocwW400cohIfvRhWa6UjpO0q2u62XwqNSDRcKPuPnHxrPMwKWta1iNJL59+6ik4g9DqyGQ3JjmAruGjIyN4GIpMtlQl69xJUzXcfWssRaWKWYa3YFyOLvfYka/RHNR77w3cL0dz8J6h/WpqSGobsl0zutAee2TaYQ9UkeLjSZ+wpUlPq2Z2b5hmGui2sAUOZRPNbK0vjwsqQZf4RPW9csxWym7HVQTfvDqOThK5GqQTutcEd8zxnIoycI/BZAPxSiTBm6uDppiwTMpTZlkDKUi3rHA7GTR776AimizKPCC6tL7+36/jr33v61++fvX1r56FdvGIe0Ex1hHvolyO3QKQZN0B3UHWWtiIr2USkSVRgbF1mrA0pVtXEqMu4OTiZoCVgJEG2LNUxlzTto2+bwv0eWUcBa8Uo/u9wGCDZwu78EH347PsSmWrqdG0YQ3Rg/LRNEIF6gtbma8m5cm2QMREeGgWayVTduNMRvQOShXQxmBjfw/yhN7RRjZak7VsjGQ7PpJtbyTbPSOpvyGyqH6x1rniTDptQPpEJQJzNVYpKEhy2IpjpF8YauWHKcsqAZ4JX+Zhri7nAGLQYImkvrOtNhsjXnMEJjy0JC+vInSho50pPvFOQLw8Tvbobso4AlsFWcT0igO5i6Or0j3kCfgvqywS/eOwXfdXutxN+L/re+SrYcYIelsBdSXCamFDywH/S0cD1OjOsznzod/e5hbVMgYPqm9Hq1qY+ZdN5e6KFkWqw0cqLKRFh5qYUCdIXXNqE9zBhamZmjjFbGDXKryNTaP8cWhTrS2TPG4mKQS9ywzx3LBi5oaRu/tveHOnFKbs9RuK3XG6kY0uS12jLQEyXCTaHTUsi/DoyI519P9XYABikwSz1//+8pzo2G/oWKPcZrT3W5h/1+KMLpDE2VTKj2kHzUt5C2UgXHnhqX780FtGfeNnqWv0Nu6iKV49GFMr3mU0vmvabFjZp1M2cC8iTOHacj79fInUW0Fq5qKGPUVGmMdgOCYVpN07HfuMG3YDOKqTzuWOUM0xxlFv0FFmBsmYhXIl6xAq90K+kDbOtLlDySMbaUFX6thl7bPMR9y1ppWC2LQjmNCqraHWqamxjO4DFs8O1U1MyphJ+L1Q7jBEdS/uQu5RyquPF+cXlnyiM41ZhQWU09NisqUWxKkimDCTRIK8hAEzPFspkgJzA3ixCsOFynZXR94WmIqvBfQxo7pPEQo/GBFGyAjfvc5DWEFkGmdo4+eHJxgwiTMOClQ6QSA5NGK6JeID4+4Bs9zbMOvcm+phSjR1fpsNwJ+TiXOGx7WNq+6a7Qiw7xayeZgtbrMpkd1C4ulK7BZXtRSR9wkvgm1dQBbBQhoGHbMDbP6w2HhSqtrQskP9g97KVomVquzLlZYTt/tZUfbZZUiNpe5gun0svHVGXKztENCtM3JBYxJgcE6RFRkfZOODtP3w0akZPtzSRlbn6z0hgqCgFheXDoy+xPMKo14098hYgV+gIJteNQiICviqbGHsq1dldFgXViMEJC2Q1K97qb+NRKMA/3JTkZDZq2Iw9ZfgaY05LiykJTnYJOjfidXc8nkPEcAteefnZuTS1I0SENafyZ8JBvC2hqS4B7N/1n0+cj0GknBG/fXaBSAn6bJ7dUTHGM1jDOUpuzZcJyksgcDt0kK/z1dxWrBn3ynYUnY/duqtYGC4kW0b8kXT8r0gzn1HY+Cn3piDKeaiGLvIohOex5pU9l+IpnAVi3KT+u2RD33ltMPEMUp3EPEo3OHFMoPi6rp09+9h9gJ+21CDCHzafnl57p2cvjh7+u70ZEq3Y6DYj8CKixZzIrzb/o47uT2Y4iN8pGC8zAO3O8nK7q87AsVBaj3++u7N29MTWjoXLzDEwdq25cbZQ3SgyAfC38qhkmYr2GOd3Gd93QZn1Kt9q5ilz3cB3v7QYkAmRuKbEdHIBZjuK4VI5CGVM/ZLWNINIRGe+saNszrMc/AT41uMI6Pr7QJG8RYR3JRUkRtq/+rl2Smg5PUp3hz/oT2KqNycY3ZhW3RZKiCDAv+WAn9MB5B5E8eXlYatHxvcWTARIlGiYe7ixRwtGLKMCAo52nW9MUgw+Jtdcx81gXuigwNGW1URwZM1RgFR8Cqvz1IipCNzflRBXhczCiXmmfwzEsoBQTB4Q0gLpv4wihoZHvDr3SWP48UxPmHqEQ7oVFbAGf189ozVn+8N71osFTW7eKmIhUDwspk8lbeMwvf7AwNjxQ2WNwSsM7gXz/52DzhZKG45Yft7JINzepV/VKIChN8HQr3w3+PCv4IFaMnh3vOIUh9BPHkMIH6XaamikGjBJNazNyfOvYEpGQH+AFmsT6Xdo7Gi5WjbCspBHeCs3/GDwVeiWIFLPzyQQQo+1OA0vCRpeAyKuiuDyApjVMQIOI+3dbwWR0KDxim19fbDW4dAX0ePZ7RQPXDN1xEgcsQ1Rn+XIY6aFRUyP7bpmspwqa9D+NwAgxZQGDCttBxFisYs8BiGJRMR6zzFJjWR8g67iYmrGO8bulCx7Wdnp69PTvHvKxyQqjk4CPiqTPPdAe2vVxkVUvx4Ro/n/R2lodBhHZJBVs12ZMgdHMgOQlxdQg+8AIYsiacuVPgIjS2FP3iTKLToULctXyRYiVP1Qb44Q+CaoIzEfV1C2YmycfMuAde5iPKjehsE3awZTnFeb/SqELvc7JXoO2hFNI12LEMLe8f1tdQ6tzF4OtNJtlSrNskfoHrT7qc2it8qcryhr/OHDrJc3rOnjiC652SeA61g/z6sz0L/+PL8lMrBBtTLf3138ua3d3ftOGKURKXWLquAjm3O27GhoWr8WQA0uHDAaHNluStH08u8sp7SuXIW5DwWFKykNqqtTBaRZpb1Aq+qGbDTYrJXwRNEn9HODYjM/tN/4MBiILRBbzeWm2UtG8q7OnFN6CYRJXCHPbW2lY2JQJQH5B6BUUtJ/C7e7ZzZcwczgR7ShV8DcnHACwQeL8Ok42TQ4tfH8Ak5uTpn3o+eKxEi740dV+o6use1dV1rAPV1R7ajB4O1PY8dv2Pbowtp5G7GHjFcoIxxumvXH6G6qcG+ErtpnVhl3toQF2PhDXAX2E8LSdnYEcCocTF+DafiA7o5YPwQk0Y43Wwv4BXK8dLYPl2XN7jJCa6yJaU1EYMReM3wT4+g7MIDC7gtOBtJ2JTQLg6OZrPZcf9Yifw8mUy809c/v3x9Wi/IZIK6EuSVJ+jvPtndP+q0jtIlj5hqNlHDqqH0Uopb8KctvCYCoAZTA/t/vZ8TEQ==", "blender_rig.py": "...", "app.py": "eNq9fV1vG8l24Lt+RW0bAzZtskVSH7YoU1mN7bENjO8osu7cRTgEUewukm01u3v6g5JMcJGXBFhgNxdIHgIECZIAQfKyDwnykH2+95/4D2x+wp5zqqq/2KSkmTs7F5fqrq46derU+aw6VTYM472fiGjKP7MbMWHm24g7btBkjvCYiJMUXvrw4idRQGV+kIhJEFyzgNnBImAhn7k+3wvTiefaHMp8/H8S8Vj4nHEGT0sRQ1OW+ixJfQABz6+8IHWmHo+EZRjGnrsIgyhhMy+Y6OdPceDr5yDWTxH3nWCRvQn9FAs7EklWLU4nYRTYIs5KknkkYGD+LCtwF2Iv65gGzXgMT3t70yhYMIupbzCaqTtThcKH0Qr9aSaSsSzZ2zv/1fsPH9mADY1XfAEUiYwWM14FUSTo6SP3Uofrx0Q+fc1dTz5dijCIA2O0d3H55uObKwS02mPw3xMg+GISiT4zQx4HcYsFdhIJ0WI2Bxq3WCJukzTiTYt1O4yqwMwshA9/B2wRwJQF7CqNJgGBM+jRAGhHLXbw4rAFrfC/Fut1Dl80W7LSJQ9dh2r1oFrv6LjFDjdrncMwsM5hR4F6sVnnA791F1TrCL4cdXvQ4ZGsdtg5OYZq6703/+38w8W3bz6OX7+/BJyNfeQfYLj9d6l/l3L/4HW7t8/jGOZ3X9zyReiJMQCdidjY29tzxJSNp64nYtNp9qnXIHJERFMhxz/mPtR3AmvmTZDYqhTYvVISuTNrOrnNS6yJJ3ynUENS2/2cQaMOs/90NRAcnrh2YAWTT4XWWfEi8QrFwE2RiN3AP3CsGL9UgIau+MxjjWsoYI71C49s1/O4foUBjF1/GlgoPcBN2HyKjDCEJiFP5tanwPVNp8WmTTYNIjZlrq/o5U6ZriRu3TiJzc02zQzkM4A5zUHEIA7CMVGALfzZaGs8NZrNZnFg0N8UtQk2B4Ag2PQ2BRlz4hs3mZumYXnBzGjpbkHC0wgr62mP08WCR3cmDlnNPYfBEgVANk2Df0rjBPikxVZr2bkHwkoEyVCZGk+frrImDRKrRos1/qjR7LfWUsyePmW/+w8tamzFZVV4bzTXLLzFj0YBoIh/TIUnkoAVAMPcYG2zUDRPcS6hsw58kC8tRvPLQL+UQBZahapRow3wmth34WMsZimoyHicBAn31DjWLC4w1dT4JogWvM9WjUXqJe4SJps3cD5yMIUPTSa8WLBGF2Y7CRrrElpmXddTBE+9EnZXimzvz/vlITVit9KtovDYzbv1scvablTlvKMi8EvgIhH1SxNARZpwJYJ8+e2/sStXgFaP+4zQZ9sHltO0NBhmkm5tVoeEhdloYCxFRtoxqqyX6si+9mgctY0nXjbG+sYrs7F0HQEM1mDPWJxEZg0UWaPRbEKVBsAwBQi4/MhCj/sBURvGVBroRvNsyEZJ1ogxH8KyT5+qOZLSX+zMII1ug8oE4VaSn0m3xcMQyGAa53kl4DwYDKpIqZEQ0rAEZaS0U6kbL3DGpAC2d/Ltd6+hZCaiYKOTkhoH2l8jIy8zpdI0SH9et9gSdaBEKe9xZLmJWMRmsw6xBQdTP46E54ql2I7cpazQZ1gf/S4fORhsJbPRHUKPLoZSj2uWNLK+jKWIwFaJeBy7/hh1DnQXAmYwU4bGd0u/U+MCdVQfnQv2FZM9QL/wG7uzVDCOvqXWkUbZ3FWVy7CxA5PGaM30Z5aVNtUwkLw3GlfFOKDUAjQIw1FzK/Zf/uYf/u//+S1b3awVIGV5jB/8H3w1udSkCZbo8v1bcteM8zRBn8ng+BeY4F26AFnBkjk9gViQTxi5nEgNH+yUO1EaCocafARMM7LgZx/81RRArJXBW7riRkTjebLwTGfwK+DbFpsLdzZPBseHHTUe8KffSL+dHbxmJjq+AvwBEEcfXXB3GvGFAHmOQE4LDABMF4qEk48P3gjof/Rx0DlXDIHG2QFfgR7u8RQMFTlYiCoY/pzSipJA5JeOuwQNdOeJQUMOor+Sf9fh7anjxqBq7vpTT9yechAwv03i0LcFhiunm0xjoK13p3dt5T7qmgih7biRsBOQ9L4deOnCP53xsN/tQEcTbl/PogA0UP9Jt9c96L6ogz0hH6mNYUIa97uH0BAABVH/yQk/cTg/nUKn7Sl4/t5dP74Dp2PRTt3GWQ2o4ripVex+Fv3D4/C2cfaff/8v/+vlPlQ4w1pnH9MJhk2cLC+7Y2HqxZzVwZxoiAor8UKcCNE4eyt8EfEIOQEjs/fnoKDeaBZ7uT85U73Rr2L2OLKBn6fGvoyKxhAM7KODPVhVp7lRnGbQ2X+UDEBkExNjKwt/QHkp7V+Yd82BkT1orOB33dDI37hOModp6Xx1uskRcgr6ncr01E3N5pw2QOV4wQ1QPPW8GJSf8E/ZbdSOYTwu99oQpdrXIG2NCnDZLG8ElJLoV2a2zM6oTNvEtP0I0T8FLxWixHYShP1DnOeXnIFYTrPxQzwI2mnQAAPO/etGZYqN8ty+mLyYTKenOe90DxDm+SRyIxTykPsJ6VqIzyFWApn+8ud//XKf57Os3OcgTcIUpBdmEhVkC5SC0DFUCCzg8cXEAeaD6K0mfHhQvFCn2tHT11Vh1DiEuqbsDCJJsB/kRaCuk8BAawFq+J6p+GIEAoA8QAbkHcLBgt7BaAOCKx4lMrYwyEsZG01Cx5i5UXBw3DGygISKQ3NaAJH3voktVfqMRHPQBbA+u6GcQ4pModgcfg5HNTT7HCo/CewRtMwi2ZLQlPR+s4VYQJygx6BwtxbhoQEfCUKrEhypud1TUx9GwQyUfCxBAkPbLbaIZy2WQGDuAG0H33BAqsVEFAWRsjTA1BEfGGCp0He4o1LNLXaCqw381oT2CyBKFyN8VAUkhdQDe4ruQFO7M2BmBhIQUkU9xER/BCyJUlAkrA3IyZYJBnkmANhnCrDZhc/43CS+pNIz1rE6B1lc6WRAM94BecL1hifi6PDFoWNgSxqvrGYaT7rTE+d4Qh/y5saTo8nRZNpR2jK5xaGj0/B3/5OtqP26AmqvpCi+/O2fsQ9yXQY5NUCRXQmvb3WmGFMAEwAwiuqI4AoYPqr+jWYJoTL0FUzjWrauATnFpSef/fcV0DD7iuCBpBvEh46ae9ttdlHJ9jq9Xm+iVXQXIuI48EBUnvR4zzk4Pq3o6B7o6Kr+DLmDy3NklBlp8VqTeloxcLvUcMmHqHoHoPlt0Z6I5AZtgVLQkyBJgkWfbPEGZGjhn61gvtcv9+k5t7q5Lu51MtdgBX/WjbMVCMf6q9zW7sJYWb1Nx+R590XXqZARsDwNwPmdgoXqz13HEf5uekjzKvE5zboCY1voipA+BWbzY5f8JWrErMN4Y8ZWjUbGiMjpuegUeKNNK3V99JZ51CaPAqhvnnQcMWspSWo9OZk8B5Om35sFhDRVO18xQpVCNkKMh4sl61pHMZPA0Ya5PjiJpw2gesmvKVvTl0SOs/96Le7IlMcMQa1wgXdV6DgMFAU6rLNOgtpPbUKss0aOIKAVEzsj9ysRoACBQ1oMYbSYJ6bwTJ4B6OW5EPrPmOYX9esCJnY8AYUMbB/mi75lczoFDpYrwFK9t2h5fCwfZ6nrcN8W1JEOE+AFYuxEkAkRQQWeNCcRwoxpmRWBg+QnfOIJvdKMAGVUOQ787FnZAdK9MFJUJqRIUPvS0HRJIRLgLrDKLLLeIOOYBjm7EBjKtWtPOb0QnEwjdOTBByFXeB58okiWFqfippVHrNng8z7wDc0M7RpY+AdtEq5RP316gDajq4yRP4Nq+XK+2awdTj+n1RPUcEEFm6zfEP0MgAjQLNBBbjImIpibk517SLKngWwrg1Uq0l55QkHmFOJCsGu4eIwR4oWMeX0ZQMLUQaG01gbIo3pZS0PVAQAFg7pHpdobAA4Fy13wduJkCJ2NlFlXb+BdeAFPoHLBtcNvC1kzh3kTRNdmAVwS3ZU9KWyF+I4UmaLU/8lSUr+AUOqsKEVacChUEbdNLT2ZyNwLD2N+miKSLfhpbpOwglDdj+UOobu3sRLEgZwgLZbSt9LiKrUzss+94GjZcoAUIto12csBKOBW5jkO8OF+MIE/Rn6GEGugogmnD7NhjUFQEwzox2PTUDWAg50CW4lbW4QJe0N/QNni7pwo85DeygN3TyDTlD5mpVYYwTjGAM9sbrAgSgmyIC6FCiUTYO8G+YahdUVPporRkLEBUQ7M5w+uolSFJeDRU3ShurgWAoMBE9RbGjpoAZqtJjiqh1KW58GND5/JyaaSmzl47QjEjccQOC6FWY5dgGSkETSpmplfS7DKZHlS8i+B3qBXJ5Fgdxx9zXxfFzx1QTu1WEOujrufwT30QP/KRdwyuRTWOOjShztXeA4zS2EK0lYjO8KxP9skxgF7VsdApvH0aXEAT5+yL3/6V4h9mAoHzDX4PB4uh8CzDZPC8U2uY725RYbgkbVlUdG4EhFuzuJKdj7KL3/6z8B7RexqG5eDpkwjKvWHsVNxhwtFrV9DJuKMCjFKfT+on1yvqii0Yx1lNivj67z/n9k3Bn0YGspwMINfwEPGCDJcIHFS2l0Kx1jGd4P6gG45BTWJ5jKSTC5V5jRFxYcRbakYt1ChFD3OlTRsQYqmMVvNiIYGRP+INTxh8IuPpMgwAs6ItJxmTgqAWiqLi85scTld0g17ALKVidPNyUIRM+mCVi1FMoFEcRMz7gSnBfaTa+I+q+6y7P0EBHSUPjUUbFqPW8bDxrQxUhH4+itD4ynVDqqRbQSgCYs9YByza3Wae1WZr8r08SaeJSGoR1ouKCg2KUufHs9//v1f/G+2a1AU9KphNCAKxo0k3OcDvJo/lU/qR/kMQQ27/cMRDhafD0fb9cdDJq08fr3SUhwyacKY9tBlaDHhiT031WIPuSHK4cgsNCB0oZ7NZu6g0yY7NrvHFV/yyAXTgF54bOx0kz+76Fx5AX4cjnBrJVuac1tydU746UJFQth1gcE0uqbL9sHn81WFFmn3Ae5V0zY6ULq73l/lFfTWjHS3yWRPaY8sdn3QEeDGmVNUmpFaX5taPsR62/3RqOCHIjwi6qDO0SM6D+i37E8gHfQmEvAUrgGOyjWARoVdpr/9M5YtqU8oXwr0Ir411+zLn/9lzcdo2ACojVFp9A9wlMr94kLVln77bCUqm15ydktbX8FMc+HibkxJLLGZ7z695R44EhQWyQSXGDxrHjm4M4drmTA7MOtiCVWCSpDXzDeccLOH+CnjJef+RBOZq2V99+srTCZSWScoXuAyxEpFl/yq4l6WG8tV4zLpcL3I9Qsej7tAPvdBRk1zV1YNGiwY6lgO1Qr9GYZosowCHSqqXSF/2No6DCxfFtc7xItZxUVGOuq5N+Fzi21MvdMc9rtHIwtiSQ98ZtMYI6YsS9hRjECghv3jzkhNfgBA5fSbYpn0UYF8FJ6wk9c84YqMQFKaRucPOYV7W2ilZ5CcY+7fmfcTMZuve0OZ3D190LyqnCVcfkAaDIFClgsu9u1or4b3Hr2PqhMd52nienkoTq+WHYR3ZVj6BVChKR/TRsN43NzoA4MwiSGYRFxyUKsHRX2pPmGSmQXhpmMiJ2wiX85Gyxb+K+qqYAp4HBcZbnODaktmlyxCuw3v3d7BYWYl52g2ozutnT4jLz6W/2gzp54HtYoEsTjKxAKsWZBGtsh0Ypl2cuLCuDRxOpINInueFS4XuBxEFa2lGyUp98YLiDtxPFklVEbUzLJTh1vwfYwGGillUvhf+IjB5RJzTiceOuBym6OD6We5Makz8OVNgJUiERroyJ0IU1hhc4324fL8AxiQ5cJKY+GAOcdVtk7f6k7X8LIZkQEoqCvzgbLKuCnx9mtMlFlYoYhwE3/9FSUcvb34NUtj7vB+PSxzNuyOIKqYDTujZrX3FX2s9lLKjspBZfkPMTrlNrfnAsaFvoew6C2zv7W293Yj3YFWF26RKcfv3n+8opW8xOiTt2TMwlQ/LiO+0M+FRzuvATo4pOd1lgRJq0aa1zbZCxlkiUyCCxLhDfx0rM4mX6JA52nT5AYZPjg5Lm/HC5e12z+mIrprA7IDBO1+plV4C95bkidp0vUzTWrL2KVUaSjoE0KUR2DC4EZElhPxG+iNUu2SgR0vW34wB7dYRPCQ+m4SGzuXneK58DzlVts8ROhKj6hCXFHTjxDcwLfBQdOKEweeMsC4wziUS2m30kbcUqYs1Ivc0GzKVV0TvCKjCeymX1uGUvtEeHAVoY+wJanOoR6lW0PEwiUz4soqH/awGNQLHx6M8PcwB4H8MJY7jMDgqlW30zsktt5fYeOsRPK0cY+SLYA0wFVGqZJN7tU2S+7FZM3sxEQ2wPEBsMES9x3gTyavQHj4rEDB41iV4wIDTvogCfP8rOtxS9IWweuUtxxdkpfh9XikfZjbXCVhqlL2fdhvn3RKecIIsJWNF54WOBPacQ2AlQK1WFXwXSM+xWMMIPZLdxkw86TD4qbFLoNPAbNTWjji3gyPPcQyV8pzF4A0iM+c3+GOMs9c2JJ41SOD8aiW370H+PI6A7q0rVdKRGqcfZAjA/eC4QYovIG2J7++uBu2JZT7FExiWc4T1Fk+TLKBhWhoDW4nQBTcY8ANCHD+cLz4ppKrv33/ocggJ88Vh5z0JIucdCRrnBwpRnhxoDY3IGCQadmmgQyptGIr32yHR1y2AO/re7AyhtaVWENuHuoae1VvTVXXtdWefg7vlezPVv2JDu8cdLZCu0L9q/Uw1j+cTg6cE3z83b++0uL/mxZ7B6M5xiyIE6ls0cMADrDnEE6V42QMl5JyqIM+WotEAyjcYim5zUSjnBcw4s24P1fjSzr6MCcXeD5sg8Y4G+DElGrZ1bwHakb+AHRYDCjwQzmiIHS1NCKipQAEjeQcc3e65VYh7YcZOjUWtNlvQA9izP/ihJRZa/UObPc7KMT8kVvSlE3SbdKG77QlaqnhtrzUAMM/6fRHlSwkmog8Gn4ZBt4d7iCzEBBL4kFjBbiuG7hG4g0aPkRYmJ8WBdcgaitM1tqBydSQNdu0aw71e9ZRmbxd63jd2D8rxH3EEQV8MLVAi7ZOijgIb9lJlgNXyqMo7N1vTaRdNdRcHxw0yug0VIZGY50lLaw30zi2A7a3gWucrZA31uzl5GxFfEgGapWuZSYEJVAoKqAWwXAC1Iz05LVmyTxT6IQqnbFOzlYlRiS31ONrctiwrQL9SxJ3k7CE5BbC6qobFVUqy7ouWxT6uIr4hH8CVxQpmdGooXSxPDyCNEVnlsBiPaLFJqXBWI2J19GfeElPt91Bo9Ngtz3g1t8Ad9/B+wrFsGN18bVXetWCoMZydKSL2g6P5zyK+N2gccgOkcWpR3TdcCOsJkVVUuPsy9/8A/tIGdFOUM2gpynGBVcDxyITn6S6/KG00cLKM6t6kKoeevjr37KrACy3zH7XJPmZuU3lCZVYVae+mjJTw2KaC19g2lNPpT1RSif7pZOeKDP6JuJhH38oJZoyn1Y4aeva/KctPgfaZAj8lHdD+WeypBoKggrY38gj2hoE4op+eCNff6OTruqTqOLljNLjvw5ugZ9ZhwEzs9U7YFk8WyeipTiPQ2Enlxi25Gp9S9rx805NhprMEUMSsQ4QScvSetVoSG4ly4LCCMg8fI4U2SuzAR1osKTDmutKpvbe3quPeO4Avc1JgLsGlszXpsnmgFjEVps5ZOy/yAiRQ1i9V9MCQpK2osdRB1Px8vqnrMSG670ncydi8y5bFdLfkIOZolRHVQnZqpwsryv0kJQMq8WuIyroTrpO7/CUbc0pZBVhQhqyYhYhgk347BeBi+haEy/ARBXVi3wtdyZ1R4mGCroiiOyzNClPZpgQWoByT+7cc/vQmU6bdRM1nU6LxXvaWcoSx59XJnhTP1U+37ZByzvBDSbFsd4hfFd4HB0B5pE7iyEkspCB2Uox9prpco9P4HdV7qRXZrL13jQIEuRELSQoqmUCLTDxDH/ZU/hLiSjyD5ipVQAqzk3u+t2NNhIvXVUhubUXlCu105a6njNOXR0mbgmdLtQGUqjyQcSC1tIiix5j62seCzOMXEzKHs9TMTBc33FnmLjlizSJINym0tjDDe7WbnfXTwbDHPTbIJh54hsoNeVpfoxsjMx8oD2NuR+3MYSYGqOmFYMHkXWA2mOc89uYHF9D6QtoW/d97PDoulApB4ZCUANNSgJCq6ugwelaVXDEMWPiaqxFQpPDKnzNAKkq5UHi0k8GRWoxPb78kwahv1dwISYuQ5JaLcMncRNP1PdVWEEP02QX0WsrbKd6miSBP44FaHIH+atmAuze9ICQrNbdjel9oDOKPwD+VtLac2Ffg3ZR1K3BXk/57ppbp7/SrJ4Rtlaq4C3lfSIoJVPdqCDTKdxkjhL/NfJBbBIjDIzzC5n+gYewcDcbt/9Rbgf0C93G8QBsea3Ef4oHhtkcnK2cwIaw1k8s5FbL9ngcf+vGEP44jtlA9CjVAZeLHLEonMvUKF0GN6bwxGLsOgMDjLJR2eaEKh8ACqh2DM6fsCrSP/gby/4XpV3oUkc/4uKhOplISW7NaoZa+5H/4akmwMb9/GPqisgJSuB036/obJ8Z2+CNDw7oJMpYRuIHPcxe0uNH810lQBHOFZ+UFiG31DENdbzO2FK3mNYLTd5jKrxJzDWQeQ3ZxjfyxF0IzBK6ntF68F4ko+x62oMdGJdvvz43sgOhvYNOcyscPYhz2wbl6Qa+aXz4/T/GKo2ZmQJcbc/heBlBpFJMMNO5CeBxq692Trf1gQxxT1USJ5DkGjK9kZiU6bNj2N3ejmFn6RBiWjclFzRY5DHrD9sfJTFv7xCs1x+wQ5luv9nZu1KqeqW/RwEf6wtWoIvXURCS0riXCEOZzt6ivG5K8SayoHMiP8hUb/qCdR4iBqoltsnhZB1gEb1U4Ruj+2EvuYfe2DacFVG/A0r46oA7SojxKJnLTuIy89KdZdJVSVPZwlGS/pcYwJmYHWZirhKmJ0nEz+UpdEzalfZMHel7uG5hucKUDvzjBnfu0R0XkaDM7ocOTDKiHNtHCMtEZHasXosdtVjXei7Pwww6VqebTcFCJFGWofZQZYfXJ+C06RsUHqHXCun0hOUr5TWYck9RYaVuXqDhU8bTlpsW7umlQggYdQ8Pf3Ss44wUOSXQ9ffBtnHncdR4BShio0cQgRZVkg0WVLdXZVyIPlKG3gXQgrI2jEcxoco3GUhHS1/PkifLgrk6UhdfPWPfgGM0//71h6bFjEd2wgoXVvXZkbpMC5wI4hQ8US3kiwcGO7hn+mCSK/pxaOAuLaokvB8L/+IFWJhzKkmFKQKKfEMk22h4MGo+llCK0vrGHTO8bd6D6COtdHbmZhfv/3HqJjzCKNW5j06KVrhLvwuiHpC8w+CRekhe8vL7f/fZnczdfAyjU4PKVNJVb5miza97U8gWOkQHWnZ5D7voAz+7iPA9oYKnvRkeVd8OcZIo1fQ1hWKZm1p/C4RRcIxnwkeb4X6GcXmzHV0Qe+9OpCpe60ZpVDItrjnsvxhthUtHnXfPCKYpy1vgYhNPPlKkHA+GZKlHGbXefBJQZ6thKLnxH9xYZejHu1z52XiDsuc25km4n2n+Jd3ixQ66zbgnAbxFfRLdaefsKrA5S33MKwA253iVAx0XKJzWeYzWlJecxINe5tkdYQBEl1nIxEVxo9h/K9D5WCbMA7LfQAhAadcQDqd+AjYXrweDGciZ/k/eXxTSfB9C9PMl9/EevV0Uf4R2omNqJXN5QHvILVbWqp2Rspzdn6hcL8jUmF/+xz918USlvPTrHvmWBw9L2HUxKeL4cAO/rsbvoJd7mdR8Rx90oLEMXl+yKP+W++jpPvAqxseQQeHzCi+oYovf/yPayoe4Xg+ZP3lUFyr/Kl1McAS9g8OMAh/FAu85NPB4g7BdvCxxcE+okh0HruhUdbOFVtae4EkQubv8MX0utExgvMkS/FJ0x5QndpRB/QaqkE/FuD7JvAO+PDFagt7uvgC49KPhY/KMAv/WRW0eu2zhorZIIozdzd/9a/MPMxXZkdWqC3OI+DxHTdLtEH6Hh53MgbkYGlnDhwRYO/jrUsSgwKQJfYj5VAdnSxQ8VBeXAlrqs6G5Pmcr5fmqw4+P8yu+DfBQevGUzKOWRyZjfTXMPQqWijFWBxtn0LqAkVu5bwqnc7b39OBoMVt00b7BfZB/lnt0L/QNc3tB2anA+16QiHubyxNJxZXN+1o81uLtEupN7L8XEUtjWvXD3fF9JlPdHuA2RDlmW8byE1ZURWwLmBn3AWupz4uLp7jbWWe0Fzoifnf14VuznGBZaL8I6iae9uTy1uVDc5ht+CbGZGHMv5yqNfTCJWMF8NiyBj56PAX4xaPKNbdU4SHTNK4Q3KgBW6MXyE2PH6IKHqiSl3R/sUSGgJubkpQv+77YYRaXYwgcagC91THFbjDAgxtKS0F4rY5igxkCuaAoI8GMm2eMR/YcE5VYDCpFKskHmYZa4VMH9PUlEzwMvbuxXIuQqYiVOyZgEkCBAq8AxtoFwnrlAEQn6evaGIonha5g3JSyiUzTUtPRksRs5URpKa6Rt0mM9spLJZY95/5MmEWUW0z/HdZfUAGRceEAIegTy/Zc+9rU18rUUXL4i1w188tcOPOzr52pdTNoup6xIWqCEWbQ+nYagYW378aUrI2eCt7iICJyCYrfsxh4r27pm2IndbjWkBcuFVQCqiK1KH6Fj+Xz0vQV1BXMXlEzttRtLZTSUN9Di4ExTkU1XKNgVPGDvFqjz8zCccxWfvapqXsZQvjZkoFdga+g0Irp3J6Zn+jTTZCYBRYsdEoHkAHmRLM/+Rkt7RSM6BM0b0lLvIXShQNGBdjZESqNBhnAYub/IpAnz3760AkGkdYEXcJBxYzvY5UFvx2TtT4+LJ3/QlB4td7jjXDJIHsYEEd7449vLr9/c/lRHr7TN9yDTI4xOcXEn/zAwitXRBADeAFyCS4VcGYLz+EELGDcT0Bd37FJGtMCA16hESV4scZE/dMLxcOEgX0tkvxkhjzF6vqYgIQLljdiUrT7GFhpXK0wCM3rmgOpTvFeuX7lgpa7TaPnAAcEsah4N9uPs2ycHUyqWBlJ6vsC9yBy5PBYXD1iiZXIK0ISjQOSIkRSRKTAkVItRvR6xrJLdzNzLmloyT8mbcjHDxh2bE1cPKBrdCz6Hwa6NX6J4rewjjjffaTT+5uwSyeYNQTAX7GWx1Pfno/lNNPwBs9fHOuB0UAHG/yXSRB8y/OjcrmSQE1KvIzGaHIHhaGpYupL0lP2ro5lYSKx2LZdhatXwk/G8n6eMWZaZM1AfcobSmSB3tTs1i1y0FqYcMaUtDkYVo99FlcvlcLQXDXUIoH3BpHob6oCklk7+0dOnCx37FaU/omLQhVjx7lgaFa8mqt8TA9U1EzQPmr7R/r9Dpd0b3ds9RnzJAnj/v7+DFg2nVh2sChgUkRqHwyt4LGI9zErLU720RNGxbu/fXvFKLRvA6HS2zZfOMeHZGxxFaZyiBvGas9Bd+MoW6wTPD86KmlXKC6zKugh/MdQYtDMDt1HmgJDDQyOB3I0/3bLF17o6uXDxliC69jyH4+xkuAazF8aeTGfCvOk+f9TAIo8z3iazAcmjqqV4VmzF4TVxgtgAwjIatOMHi4rP0s8pGEkh13Gsr1OZzExqlKD5qMkMkDp4lW+JcUoBaUsQsU7Q+zycdULOoE+JA7KFb7RbvtBG+9jlzezyCLo9iExyJSkBISk23tOE9ftr3Cu1g9Z3JLHSQdFDN9fvEEfG7zuqFj+8eo1ULVwLLXgM1Yu7MsMjbwsrHyVEnvJjjuVWz/kSQ/sSZ1vtZAPvPLBbgrd6S5e8AIxWjOjTD8Meftzp33SHj37wYLpyWfjB9QZuBbiyvuTqrExiN1i0xLJCV9YmLEXmjURJjgl/Lpwt1X9FWza5xvS7U1on8donwvjHJVvaCtdzVZmSsUrI0WnhxzCpOvkTCP/J6wqRy1R+2ZuBg554+pLBeHXMa1n4BaPjxdvMPmPaNECkfxnuOQ/sRXJdKzAsiyj4gNXnaV6jbShlUjZSE1Qr2u2q45a1n+8Mc3ZgRAjjFD11kzRht5QdgFqS9WfI773/wBuwW4p", "estudio.html": "eNq1W1tv20iWfvevYJTuFplQ1M1yYsl0NrGdy8CZBHF6egDDSIpkSWJMsjQkJctWC5hfsC8L7O7boh92gX1YYIF9mLd9mPyT/gX7E/Y7VbzpYifdPRMgEVl16pxT536qmIN7nnDT6wnXxmkYHO4c0I8WsGhk13hSOzwYc+YdHoQ8ZZo7ZnHCU7s2TYeNxzUAy+GIhdyuzXx+NRFxWtNcEaU8AtiV76Vj2+Mz3+UN+WL6kZ/6LGgkLgu43SYcqZ8G/PAkSaeeL7Tu8UFTjewcJOk1/fZjIdJFo+GM+vfbnXa3/XjQaExYxAO8O22vs4v3wI94/36HdbzuHl7Tedq/zx/zfc7xFk7xts/2Pcbwxly3f7/n9JxhC2/iEliG+96es9x5sHDEvJH4N3406jsi9njcwMiSZGI6wrtehCwe+VG/NRhzfzRO++1W69uBw9zLUSymkdefsVgnTo2BKwIRZ+/gxhgMIZZ+e3cy15LrJOVhY+qbDTaZBLyhBswzPhJc+/6VmbAoaSQ89ocDMePxMBBX/bHveTxa7txPxWQxEQnkKKI+cxIRTFM+wCiYmcwHAR+m6inOOMSj5yeTgF33hwGfD0Zs0n+MQRb4o6jhg3LSd6EwHg9uGn7k8Xm/NyDIxlUMUPpnuWMlfLRYQbOxa6kSY6Dk1m/TRkXge5qaJQXlk42Yef40UbxNmOeRvLsTxVpnMlfkNGeapiIyLSeNFhVqaQzxTFgMllekDC0X1FtrlB5VCOFZa4OK0ogfjSFoYJrGCVBNhC8lcTWGYBog4/J+JEoZZExZYoWl+x23M+yyDaWvrOlDfMwJuLcQQOun132r28vpenzIpkFKCyYLkm+/jWfPK1Ud84Cl/ozL4cN1oXy9Crbw6HmaFfJoWig4EhEfbBqZsqiWMrb2Nsv/jTawh+fQj1Sw6Lf3aS5k80bmbXut2bj0CDZNhWTeEhMere3ACYR7iVka1JipfpUeVmEGGS1y5JTP04Z0iz650aYPSwCPuyJmUjJKThnzj8muiOPVDdKeKnKSSworvcMEC+b7Y9ryyhbU0BYLzBclIQuCxYZ3QG4s7Us1IpLw+eQWOxKXhZqqOGiYOG5c5fpoAY0jwFK4JSapib70NRmWeiRj8t6hiEPlx7Bp/ke9gRmjEn1WAo0MRZ4fc1eiB0PTMNoWvfLARhajtLq/N7sChyzyw2SxEQRJM6txbvBpmqT+8LqR5bAMc44iN6Df7HeVUJWbSWfND74YotaYsrbxhVS3VZFyXA3cHw6HQOVHQ7FFhSqbIGlJn98jHjeNiiwCSZMrTeda3B3QvnPXbVs9UAn4iEfe4rbIItdnVvNozfd3e+u+X/WqrBBw3V8degrvXdvOSkgs9lbsRfP82XbL2mKgWfUAtFprmSPwF3kEwqKirNgII90KL34kRZvFuPtw6m0y9SMUa4gxK9xt4Wq7zW/qOd98d7njsmjGkvVge9DMKraDxI39SapRXWnX/JDqwpBNaoeL7CWp9Re1dBxzXuvXxmk6SfrNputF1qfE44E/i62Ip81oEjYl0D+0rPZey2o1nakfeGrMCoU3DThW1EyFqglNiihp/jKUfM5C1GBJ81MSNmvLJTYhmccumqr2pcIPb1C05nt2DY6AslXT5IAbsCSxa0jyNTlJVXAipwGgHFPzWMoaMyzk81q+QES1w/dIJtOYHTQV3OE6PEocCOrwrc9vWFJAbUWt4jEB80Qkt2IE8eva4dPY9YOAVTEeNLGZjU1hn1snPE9tNiCitYJKNouEUjv8+V//rJ1Ob7Sf//l/S24qKChJbRNTQF2H7ARqeUuwfTcEOIeZ+iIGZPak6TB94w5RYRV+4FYux7Kn+eOtJLyYhcj0LqsdHuPx80/0fCs0fCEC5Fv60fQYfU/CYi1Vai7ZUjK9U7IuC8Ptgv2/f/vHf9eOPv+EUMJ+sXRdu9YyW7XD51Q78+37AEz7sYQ6QfUbeOxWsH0Jdco8ofk3f7LuEDuAGxVoRDXrVqzdntlGb9lt7t4K0jIft8iMY9+5nbshKvfa4RGrOtiXBY+SaLvcT+YUt6DOO2We41AKWKGX/+RhhIK25G/EIsgE0YwH4uc//8cGnMwShScW45SwN0dV8izjE43JKuEWVy4CVypEsB64aDwW6UrQeorE28Ag23Qaabgp/OoI1hVXADaQXqGYg5+I4PN/jUS0JWARUHLJAzLCP015wFOxHWYM9uAT//QX7YhNVqJpGdXyn5WkpLIHNqwSkvZAY4n2/uW7kxNtGItQq8ssUR/k84s3seOnR0iRMQS1rALlecfNJpsroMg/FSwvTt8/P4XmUb5tQxHIqaRZgq0uP3739OjNV6yvwK0ieCdEeBLN/FhEsNF0KxJezifNtQUK2w62mqTaN3ZiH3rCncoZ6Cm+PoOu3FTEemIMMqjnr05PzuwFwmD/vK7M/MMVd6xR4NTNfIBsFKFhdVBFTv8mn7gwZUoEmonMidlglvxomLJfNkq5DkNMJTs1uMx5+v2roxN7sc5Nv/7i9JmGrMZjoWFQ0xH0XaE91H739uSFsZ1btSgb0XQsM7UXwhOpqb2XdcqnpLISda41dOZY9fzZH+mkTOO5gWv695GfXpva91HMWWBqr32UJgKLd7RsueUEiAVY/Ix+OfKdGjFul5liL/ErhEpgpFrKbsISzidAvnn2O0ClMsuJCtkPsJ6YJyguu56VpAFAz96fatno5/+JtO6xpodhhY1AeO0PvVbGwumb41ysaPRWoDof2r0KVDi9ziHbvW8lDxVd9+uqGNJgzNp4Cm0DV6n0fv01mzCkF02OYa6qfTCNilaj1pNhauZ7XHxAovUjFlvhZBcAf6AxpUuX9lWAjWD/XZSLVTC8//W/JQSk+GFC6Z5Zk2gEgFchQ9AmRuSkpibry8JzYqk+HtsRv1JRx/qBOy9O32Xj+oJFdE7qswRt8pSbJGkezzgM8grdyrPpcIgmh6aWcLQcnYWC/60/58E7Op7QX7N0bGF/ujqCLWfMjmEMqovO0O7ofhTx+IfsmBaPL6VfVdGLaTqZIrKhMTijkzFbsX727sWzcrBEjDDMoZEJGM4gnx6dnD33g9B335dzFU7GMNsrDFs8kidlNu1wsFPEGCrFLazCgqMxWgG9WOohTAWcgIrAk6CR4RUJn9G7bgzkuFU2jxUQuQu9NVfHzAWmSRjzsAL29vW7k9cvgCVmFOtyJqilJnAEUNT2tlxlUYBVlGn9WjjVDRM9yK5hKbflOUGYZZUcQvqETj9m/IiFIKqjSCq11axoi9C1zU6rVfDupoFEtZKUdBAwt8oO4Jnwj9EXkeakCmiYeu53yPwp3zZ2NuHQV9vq5IRHsV8V7Qu8vuTBBMbdbpm7qAbnHd5tq4dOZ7fj5ppBGtJpcbEFdPyi6iqveTLWK+JBvc1fcBHyNL7WOy1s3zAras+MiloEFujVw9elIY/ERGxRRUPdszW3G9Jt3r5qdvI5tAgQvcKjtl4yKkEKTsc89CuMvsRrMhlDyKekHBjWUP7Blrv4s7trWnvVXdPy0nynUQXVcX7+xYI1XB1gmEaWy5J0hUWMKX+y0HqTg5OjQzy7j036p0oXsAXZ2A/vJvv4MWMg264iwCIg2EF20dwpVDUNAjNEuMkemUSR2Isl0rJwL6seR+9wA0fMK6PPxLyLwZewp8cmFYLKchQ2qnwJFZsxX9n2GTZWbEDVQHK8LKP03Dc9yuhyslIjYVaOk4SOuYvkFL+FEej1X3uOEPhO0pQom6MgHTbrxkCxJSlUCEsYEt1wGkkhabKp1i+NBQpYxfKpvch64v55qXXzvGPumt0Ls9QIrMlsWy2T9H9h5u1xtqjjeWbXwrKeCQOgZc7Qe4xlVs+09mmpWkbFetEjq7WstQenAsR5YxeE5eK91qMW0QRGOJL8q9YXLXOF2S5onHfNntlo09rhsNd79Mgk3Vpgut0zhyxIsFY20JV11q553sKynlqlXAfr9iRdtWh5fnkxANPSByh8k4hf8rl+et66UK5Bx6VRAp+3T8/bF3IoPy2TPmFZ1ul558KwwmmQ+pPg+sxlAYv1l0iSfriOtXuhhqtYdyULNLqCuNF58NKkv43uAyAj915Z1rvIfKhScmP40cUTlUL6ZO+bIK8qKPYkZX+o36N1xmILjYfW3nJnWVqYA1/TjQUc1ZpO0C4jD6exP/9BxEiopEQ6J50jalzyk3CSXhe+42x4KNEmPGj3ZkhSXBf24QLMCMtPKEob+cvZJSUpT43Bzy5lJRplDACRgJhDFBf8GaVkKnGIyYGD4ck1UDjlsEEVQHCtuN7FXFjybyxls8lhGSAzypLCF3AXYF9NhOQzjSBM3TGWS6qlkL0hWTGvyHkIT+A6u7HhGzyw2x3p00r6mURdmzCNeHokz1wrSe0Pso2CiA0zyYFklbYNZPDSVrUem+uJdW2iBlCqUSGvOjk3CSCxbowHbWu/KWdSFlFBYA3F7EGe/FDdGsiABRJms5tikg6IuM2D6oAEZWsO4FrzhxIGDYjODAXvikTnxgPPdK3rcjYbuXlYwKzDqy2h6EB0gjSU9ugeA1QjzmLba6Lwka9DenvQUy/Kxt7G4pNKZUqnuqp1CgOsegi1RQlchPJZarfMmd0a3G3mWcljl8Y0mD20RxZLQcuB6SWlYFxYWTpIaVoeqD/JftVEs9u/Y1mzS+ZWaOUrTAOw3+h1eb9TNyxVKr5/fWp//GYhpStLYD1FCSpOBX2VcQbS0Ug3lhoePv8UjaaBSLS//kX7ZjHbAjT7/J8xwj1PDpz48JsFLAxAz5H7Pb2D6c//otHg9bbBm5XB8OND8l2dOLVkc5c8+ajoVoaWqu9LPvbrdTg7S64jVysUx+fotxJ9aCyggUwpsc2umI8qkqfuWB+aC6hnLLx+/eXJ0+N66bxoby6XLiMgbiyyQZljlsuddULJWFzpdNhfydMui7zElscd5zR1MSD7GfoBl3XLAJ2nnhW0mhgqcANWJOsYa8yIb2Mh4YcDJ+bscpmFdhozFEe5PqmcgD7lhYuV3cPYdbriqZfmMcq2ntUe9POU9qETQniwNGIOLaQsMCpYqRs5yr7i+bh5RKmtmE671XrAJW7uNXNky28/KjsFAXiOsVBJDA2RmHE5ssRMWdmtAVQmBmvl31KlHDiPXPEFvxQbdbHYVs8LtGlT1FjhESigjZFqH8hUAnSkTNu263SkVDdkKpC9xFpLcoY46qFwKlsNdYWGSmvfcd09E/IajSOeJH2r1zPp26lAvbV6Slgr1LJjrTpt4qspQgxQnmxjs5ODCtFHSyQrRaks3UkXg/W8FEYysIR+ZF1LCRdR6Lphh1HWGJWDiI/Usa2OAGnZAwzKHmBAliYPpVcCUr2e2QvCH53ASHAr4NEohSIzTMX2n+Ygr2mi2EVu+fLq1D6vv+NgiU6JjtRBDz2JGE0vHs5YMPVY9pTKh2fwRPnwViT8w/v6hcK5wlEi4lTXmekY9qEu6ajw/QaebNG3cMbD/X3j2/39xtqsszKbsVsGBZeCQpWSschkdu7KlRe2FILlBv7kqZyh7Cex5Nt27OKoxEUASXnW1+t1dSpep6qn6t0KM1wC8cPlev0D9q5JKBGBDlo1Hduk4KIrUGNFe9WjGCervSRwznmugIsn+VO/ukWU6BlWtbUsG9MhhsRGARSB8QwhIPPJ29y9WmDmK7LjMWKYggj4zb/Asu9lMNVJectxigSSR6G6FBh5C09/QCOsEzDdXawA07E/86NEgRcJV93bbITode/+8cd8RJ5t15/U5WV6vV+nDw7qG9hKf1nH9KR+4BT3v4eUjY+YxzQZgzRbm0Yqc1o0cyxgk9iWFvIb7CQytYBp9MkQo0FPOAGz6n2pznX2iEh+Iy2p1B+qlK3OaX/8EXn5YV1TbybIssppbV85uaqM0RPLXhcqPVUnjVndNbg9wRVC2X7t8DQIsFRewmu5xaOsjU8YsrpjHzoVvaViNAqUkk3HomtDqNmaYa8yr8uKMDvQoGCjolhZI0ozj/LKD+VxZvHRxYBSNsvztUqAhOG777KHe7bNjOwZparH30zRhHaNAbPomDfVDTn6KqJBSxIyBjkb7Eu7r34U9LW7r4YE24424oFhVKvjzBtEJJtIua07UrD+NI7ZNV7lr14mM+NJ+dw/L58vSqZD+zC0yOOkxdigiPy1LA6apNnY9ex0pF45KN6US/7R5LpI8jhH1RCHuYkJOoUJG8kAVR4b2Y6lPjnNImo2LmxveyhAXFSx426WKtzM7cP5thgkMRnSqIRRpUbZO5umpF4QwvjJDA8ERGfUel3usG5SKP8bsWPcKetz9UXExW2yJkYWhf5K5wvWI8Lya+i4X6CzenSt0ghFmFikd0Z9n1J2zppLMZC+KUA99otb+Aw0tTcn0SGb6rRljtrq5QOr9Yi64LyYWW+oU4CnaOBTNMoA7vVUF1vtiNO1xlbFoWXRFJwzE6VEZWcWYquPEsGENEI20X8/DR0qvLPzC5N/SQ13BtyqLmTXVIm1ZEalJu7S2r3V9636q4a0VXBjqei4abxGJ89EO5XUvsqHcvPIvvcVyX+wFabKFyLYoBI/M76yAuQOwhslyhbCGzDrhJVNU9iu9lTlRUl2KJfd0siSugQswrPl8Uk6fo+gm/nSNhj6PlH9949O9Z6h0tYp+/57Zo5FaFX+q4Bq9EIru/6x5fWPzCbZceHXNqCDvzfTOYftweoG1FmE5Dm3m/Gm4xS3eupBl9syEUjyKMRu7RFYnUqQMbKtveVuEOZ0DMf9/t2pXvfpgrtJl920Am10ROUaOmT1EY68BseEZEsnK9/MSfTpwA1XSWlBcY7JO057+73m3Qd5X3uVvX5XXjSRp0JM9IrbeeiMqAynuH7M0RzqWYfy3XcrjYq3GWxvkz7R3tHlMZIkBLOjMyp5maVOafTqOVU99kcfZF39KZF9hfyFLIsTqjKio24vrsHeOCQca4belieqMGexO/ZnAqX5Yqn6k7WTqN/4WY5Z/USj+j3G6vcXFwb4VHem25jIGL/k14lOn+cYa+BKOFB+6CeIJsg755ZFvxcyaWXnc8obFWxxGGioEzaFKxPBr01nZQepDvoq2ezCSgSSCXhYOdGrHNeGNnlt9n1g/euksHFA+Au8eFh1Tnopu0dCfj68ePhRO5D/V+Twm8UwLwQsbHoChzDoY2g5+RGhqNrrMxk213iXbBK/9zLkv43Xak8y3EpfqVkWFXVAk0CXBiXz4htu9Kj08fZBU/3nxv8H1J+7UQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()